In [10]:
import os
import json
import time
import pickle
import numpy as np
import pandas as pd
from dotenv import load_dotenv
import xgboost as xgb

In [11]:
load_dotenv()

True

In [12]:
# definindo os caminhos
datasets_path = os.getenv("DATASETS_PATH")
data_path = os.getenv("DATA_PATH")
models_path = os.getenv("MODELS_PATH")

In [13]:
# carregando o dataset de transferências e aplicando os filtros
df = pd.read_csv(datasets_path + "/transfers.csv")
df = df[
    (df["transfer_fee"] > 500000)
    & (df["market_value_in_eur"] > 500000)
    & (df["transfer_date"] >= "2010-01-01")
].copy()
df["transfer_date"] = pd.to_datetime(df["transfer_date"])
df = df.sort_values("transfer_date").reset_index(drop=True)
df = df.rename(
    columns={"to_club_id": "buyer_club_id", "from_club_id": "seller_club_id"}
)

In [14]:
# carregando os clubes pré-processados para obter as ligas e features dos clubes
df_clubs = pd.read_parquet(data_path + "/clubs_processed.parquet")
cols_clubs = [
    "club_id",
    "league_tier",
    "confederation",
    "name",
    "net_transfer_record",
    "national_team_players",
    "stadium_seats",
]
df_clubs = df_clubs[cols_clubs]

# juntando informações do clube comprador
df = df.merge(
    df_clubs.add_prefix("buyer_"),
    left_on="buyer_club_id",
    right_on="buyer_club_id",
    how="left",
)

# juntando informações do clube vendedor
df = df.merge(
    df_clubs.add_prefix("seller_"),
    left_on="seller_club_id",
    right_on="seller_club_id",
    how="left",
)

In [15]:
# carregando dataset do modelo 1 para buscar as features exatas do jogador sem data leakage
df_model1 = pd.read_parquet(data_path + "/model1_dataset.parquet")
df_model1 = df_model1.sort_values("date").reset_index(drop=True)

In [16]:
# recuperando as features do jogador imediatamente antes da data de transferência via merge_asof
df = pd.merge_asof(
    df,
    df_model1.drop(columns=["market_value_in_eur", "log_market_value"]),
    left_on="transfer_date",
    right_on="date",
    by="player_id",
    direction="backward",
)

# removendo transferências que não tinham nenhum histórico de features prévias do jogador
df = df.dropna(subset=["height_in_cm"]).copy()

In [ ]:
# carregando o xgboost treinado do modelo 1 e a lista de features
model1_path = models_path + "/model1_market_value.json"
model1 = xgb.XGBRegressor()
model1.load_model(model1_path)

with open(models_path + "/feature_names_model1.pkl", "rb") as f:
    model1_features = pickle.load(f)

In [18]:
# prevendo o predicted_value_m1 para cada transferência usando as features exatas da época
X_m1 = df[model1_features].astype(float)
pred_log = model1.predict(X_m1)
df["predicted_value_m1"] = np.expm1(pred_log)

In [19]:
# otimizando o cálculo de ratios de compra e venda sem vazar dados (apenas histórico de 5 anos)
def compute_rolling_ratios(df):
    # isolando arrays do numpy
    dates = df["transfer_date"].values
    buyer_ids = df["buyer_club_id"].values
    seller_ids = df["seller_club_id"].values
    buyer_tiers = df["buyer_league_tier"].values
    seller_tiers = df["seller_league_tier"].values
    ratios = (df["transfer_fee"] / df["predicted_value_m1"]).values

    # arrays de output para os resultados das iterações
    buyer_res = np.zeros(len(df))
    seller_res = np.zeros(len(df))

    # delta de tempo fixo de 5 anos via timedelta
    five_years = np.timedelta64(5 * 365 + 1, "D")

    # iterando cada transferência para calcular sem olhar o futuro
    for i in range(len(df)):
        t_date = dates[i]
        start_date = t_date - five_years

        # busca binária para delimitar exatamente o período de 5 anos até o dia anterior
        start_idx = np.searchsorted(dates, start_date, side="left")
        end_idx = np.searchsorted(dates, t_date, side="left")

        # se o histórico for escasso na janela geral, assume valor neutro de 1.0
        if end_idx - start_idx < 5:
            buyer_res[i] = 1.0
            seller_res[i] = 1.0
            continue

        # recorta apenas os valores que cabem no período histórico válido
        window_ratios = ratios[start_idx:end_idx]

        # função interna para definir a mediana com fallback
        def get_median(ids_array, target_id, tiers_array, target_tier):
            w_ids = ids_array[start_idx:end_idx]
            mask_club = w_ids == target_id
            if np.sum(mask_club) >= 5:
                return np.median(window_ratios[mask_club])

            w_tiers = tiers_array[start_idx:end_idx]
            mask_tier = w_tiers == target_tier
            if np.sum(mask_tier) >= 5:
                return np.median(window_ratios[mask_tier])

            return np.median(window_ratios)

        # computa a métrica para o clube comprador e vendedor
        buyer_res[i] = get_median(buyer_ids, buyer_ids[i], buyer_tiers, buyer_tiers[i])
        seller_res[i] = get_median(
            seller_ids, seller_ids[i], seller_tiers, seller_tiers[i]
        )

    # colocando colunas novas no dataset principal
    df["buyer_ratio"] = buyer_res
    df["seller_ratio"] = seller_res
    return df


# executando a função sobre o dataframe carregado
df = compute_rolling_ratios(df)

In [20]:
# construindo e salvando dicionário estático de ratios para a inferência futura
max_date = df["transfer_date"].max()
start_date = max_date - pd.DateOffset(years=5)
df_recent = df[df["transfer_date"] >= start_date].copy()
df_recent["ratio"] = df_recent["transfer_fee"] / df_recent["predicted_value_m1"]

# mediana geral como último recurso de fallback
global_ratio = df_recent["ratio"].median()

# agregando medianas de compra para clubes e ligas
buyer_club_stats = df_recent.groupby("buyer_club_id")["ratio"].agg(["median", "count"])
buyer_tier_stats = df_recent.groupby("buyer_league_tier")["ratio"].agg(
    ["median", "count"]
)

# agregando medianas de venda para clubes e ligas
seller_club_stats = df_recent.groupby("seller_club_id")["ratio"].agg(
    ["median", "count"]
)
seller_tier_stats = df_recent.groupby("seller_league_tier")["ratio"].agg(
    ["median", "count"]
)


# função auxiliar para construir dicionários baseados nas estatísticas com mínimo de 5 registros
def build_dict(club_stats, tier_stats):
    res = {}
    res["tiers"] = tier_stats[tier_stats["count"] >= 5]["median"].to_dict()
    res["clubs"] = club_stats[club_stats["count"] >= 5]["median"].to_dict()
    res["global"] = global_ratio
    return res


# unificando todas as métricas num dict raiz
club_ratios = {
    "buyer": build_dict(buyer_club_stats, buyer_tier_stats),
    "seller": build_dict(seller_club_stats, seller_tier_stats),
}

# serializando dicionário em disco
with open(models_path + "/club_ratios.pkl", "wb") as f:
    pickle.dump(club_ratios, f)

In [21]:
# carregando e juntando dados adicionais fixos do jogador para calcular a idade exata
df_players = pd.read_parquet(data_path + "/players_processed.parquet")
df_players = df_players[["player_id", "date_of_birth"]]
df = df.merge(df_players, on="player_id", how="left")

In [22]:
# criando features derivadas: janela de transferência, mesma confederação, idade e target
df["transfer_window"] = (
    df["transfer_date"].dt.month.isin([1, 2]).astype(int)
)
df["same_confederation"] = (
    df["buyer_confederation"] == df["seller_confederation"]
).astype(int)
df["player_age_at_transfer"] = (
    df["transfer_date"] - pd.to_datetime(df["date_of_birth"])
).dt.days / 365.25
df["transfer_year"] = df["transfer_date"].dt.year

# padronizando variável alvo final em logaritmo
df["log_transfer_fee"] = np.log1p(df["transfer_fee"])

In [23]:
# selecionando colunas finais para o modelo 2
M2_FEATURES = [
    "player_id",
    "transfer_date",
    "transfer_fee",
    "predicted_value_m1",
    "buyer_ratio",
    "seller_ratio",
    "buyer_league_tier",
    "seller_league_tier",
    "same_confederation",
    "transfer_window",
    "transfer_year",
    "player_age_at_transfer",
    "position_rank",
    "national_team_ranking_inv",
    "goals_per_90",
    "assists_per_90",
    "buyer_net_transfer_record",
    "buyer_national_team_players",
    "buyer_stadium_seats",
    "seller_net_transfer_record",
    "seller_national_team_players",
    "seller_stadium_seats",
    "log_transfer_fee",
]

# limpando os nulos restantes
df_final = df[M2_FEATURES].dropna().copy()

In [24]:
# exportando o dataset pronto do modelo 2
df_final.to_parquet(data_path + "/model2_dataset.parquet", index=False)